# Projeto Big Data — Crimes em Chicago
**Pipeline:** Ingestão → ETL → EDA → Machine Learning com Apache Spark  
**Target:** `Arrest` (Classificação Binária — predizer se o crime resultará em prisão)

---
## Parte 1 — Ambiente e Ingestão (CSV → Parquet)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, hour, dayofweek, month,
    count, sum as spark_sum, round as spark_round, avg
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    MultilayerPerceptronClassifier
)
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

In [2]:
# Conecta ao cluster Spark (Master + 2 Workers via Docker)
spark = SparkSession.builder \
    .appName("CrimesChicago_BigData") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

Spark version: 3.5.0
Master: spark://spark-master:7077


### 1.1 Leitura dos CSVs do Data Lake Local

In [3]:
# Lê todos os CSVs da pasta /archive (suporta arquivo único ou particionado)
df_raw = spark.read.csv(
    "/home/jovyan/work/archive/",
    header=True,
    inferSchema=True
)

print(f"Total de linhas: {df_raw.count():,}")
print(f"Total de colunas: {len(df_raw.columns)}")
df_raw.printSchema()

Total de linhas: 7,941,286
Total de colunas: 23
root
 |-- _c0: integer (nullable = true)
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: string (nullable = true)
 |-- Domestic: string (nullable = true)
 |-- Beat: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Ward: string (nullable = true)
 |-- Community Area: string (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: string (nullable = true)
 |-- Y Coordinate: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Location: string (nullable = true)



In [4]:
df_raw.show(5)

+----+----+-----------+--------------------+-------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
| _c0|  ID|Case Number|                Date|              Block|IUCR|Primary Type|        Description|Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|          Updated On|    Latitude|    Longitude|            Location|
+----+----+-----------+--------------------+-------------------+----+------------+-------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+
| 388|4785|   HP610824|10/07/2008 12:39:...|    000XX E 75TH ST|0110|    HOMICIDE|FIRST DEGREE MURDER|               ALLEY|  True|   False| 323|     3.

### 1.2 Conversão para Parquet (Otimização de Storage)
Parquet é colunar e binário — reduz I/O significativamente em comparação ao CSV para workloads analíticos.

In [5]:
PARQUET_PATH = "/home/jovyan/work/data_parquet/crimes_raw"

df_raw.write.mode("overwrite").parquet(PARQUET_PATH)
print(f"Parquet salvo em: {PARQUET_PATH}")

Parquet salvo em: /home/jovyan/work/data_parquet/crimes_raw


---
## Parte 2 — EDA e Pré-processamento (ETL)

### 2.1 Leitura Otimizada a partir do Parquet

In [6]:
df = spark.read.parquet(PARQUET_PATH)
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- ID: integer (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: string (nullable = true)
 |-- Domestic: string (nullable = true)
 |-- Beat: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Ward: string (nullable = true)
 |-- Community Area: string (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: string (nullable = true)
 |-- Y Coordinate: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)
 |-- Location: string (nullable = true)



### 2.2 Limpeza dos Dados

In [7]:
# Remove linhas com nulos nas colunas críticas
df = df.dropna(subset=["Date", "Primary Type", "Location Description",
                        "Arrest", "Latitude", "Longitude", "District", "Beat"])

# Remove registros com coordenadas inválidas (0,0)
df = df.filter((col("Latitude") != 0.0) & (col("Longitude") != 0.0))

print(f"Linhas após limpeza: {df.count():,}")

Linhas após limpeza: 7,834,133


### 2.3 Feature Engineering
Extrai `Hour`, `DayOfWeek` e `Month` do timestamp do crime para aumentar o poder preditivo.

In [8]:
# Parse do timestamp (formato Chicago: "MM/dd/yyyy hh:mm:ss a")
df = df.withColumn("Date_ts", to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))

# Features temporais derivadas
df = df.withColumn("Hour",      hour("Date_ts"))
df = df.withColumn("DayOfWeek", dayofweek("Date_ts"))
df = df.withColumn("Month",     month("Date_ts"))

# Variável alvo: Arrest como inteiro (0/1)
df = df.withColumn("Arrest_label", col("Arrest").cast(IntegerType()))

# Domestic como inteiro
df = df.withColumn("Domestic_int", col("Domestic").cast(IntegerType()))

# Tipos numéricos para colunas de área/setor
df = df.withColumn("District",       col("District").cast(IntegerType()))
df = df.withColumn("Beat",           col("Beat").cast(IntegerType()))
df = df.withColumn("Community Area", col("Community Area").cast(IntegerType()))

In [ ]:
# Seleciona apenas as colunas relevantes para o modelo
FEATURE_COLS = [
    "Primary Type", "Location Description",
    "Hour", "DayOfWeek", "Month",
    "District", "Beat", "Community Area",
    "Domestic_int", "Latitude", "Longitude"
]
TARGET_COL = "Arrest_label"

df = df.select(FEATURE_COLS + [TARGET_COL]).dropna()
print(f"Dataset final: {df.count():,} linhas | {len(df.columns)} colunas")
df.show(5)

Dataset final: 0 linhas | 12 colunas


### 2.4 Análise Exploratória com Spark SQL

In [ ]:
df.createOrReplaceTempView("crimes")

In [ ]:
# Query 1: Top 10 tipos de crime por volume e taxa de prisão
spark.sql("""
    SELECT `Primary Type`,
           COUNT(*)                                          AS total_crimes,
           SUM(Arrest_label)                                 AS total_arrests,
           ROUND(SUM(Arrest_label) / COUNT(*) * 100, 2)     AS arrest_rate_pct
    FROM crimes
    GROUP BY `Primary Type`
    ORDER BY total_crimes DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Query 2: Distribuição de crimes e taxa de prisão por hora do dia
spark.sql("""
    SELECT Hour,
           COUNT(*)                             AS total_crimes,
           ROUND(AVG(Arrest_label) * 100, 2)   AS arrest_rate_pct
    FROM crimes
    GROUP BY Hour
    ORDER BY Hour
""").show(24)

In [ ]:
# Query 3: Distritos com maior volume de crimes e menor taxa de prisão (hotspots críticos)
spark.sql("""
    SELECT District,
           COUNT(*)                                         AS total_crimes,
           ROUND(SUM(Arrest_label) / COUNT(*) * 100, 2)   AS arrest_rate_pct
    FROM crimes
    WHERE District IS NOT NULL
    GROUP BY District
    HAVING COUNT(*) > 1000
    ORDER BY arrest_rate_pct ASC
    LIMIT 10
""").show()

### 2.5 Análise de Desbalanceamento da Classe `Arrest`

In [ ]:
print("Distribuição da classe Arrest:")
df.groupBy(TARGET_COL).count().orderBy(TARGET_COL).show()

count_arrest    = df.filter(col(TARGET_COL) == 1).count()
count_no_arrest = df.filter(col(TARGET_COL) == 0).count()
total           = count_arrest + count_no_arrest

print(f"Arrests (1):    {count_arrest:,}  ({count_arrest/total*100:.1f}%)")
print(f"No Arrest (0):  {count_no_arrest:,}  ({count_no_arrest/total*100:.1f}%)")
print(f"Razão de desbalanceamento: 1:{count_no_arrest//count_arrest}")

### 2.6 Balanceamento — Undersampling da Classe Majoritária
Reduz a classe `Arrest=0` para equilibrar com `Arrest=1`, evitando viés nos modelos.

In [ ]:
# Fração de amostragem para igualar as classes
fraction = count_arrest / count_no_arrest

df_majority = df.filter(col(TARGET_COL) == 0).sample(fraction=fraction, seed=42)
df_minority = df.filter(col(TARGET_COL) == 1)

df_balanced = df_majority.union(df_minority)

print(f"Dataset balanceado: {df_balanced.count():,} linhas")
df_balanced.groupBy(TARGET_COL).count().orderBy(TARGET_COL).show()

---
## Parte 3 — Machine Learning (Pipeline + Avaliação)
> Padrão baseado em `aula_06_parte_2.ipynb`: Pipeline com StringIndexer → OneHotEncoder → VectorAssembler → Modelo

### 3.1 Configuração dos Estágios de Feature Engineering

In [ ]:
# Colunas categóricas que precisam de encoding
CAT_COLS     = ["Primary Type", "Location Description"]
INDEXED_COLS = [c + "_idx" for c in CAT_COLS]
ENCODED_COLS = [c + "_enc" for c in CAT_COLS]

# Colunas numéricas prontas para uso
NUMERIC_COLS = [
    "Hour", "DayOfWeek", "Month",
    "District", "Beat", "Community Area",
    "Domestic_int", "Latitude", "Longitude"
]

# StringIndexer: transforma string em índice numérico
indexers = [
    StringIndexer(inputCol=c, outputCol=idx, handleInvalid="skip")
    for c, idx in zip(CAT_COLS, INDEXED_COLS)
]

# OneHotEncoder: transforma índice em vetor binário esparso
encoder = OneHotEncoder(
    inputCols=INDEXED_COLS,
    outputCols=ENCODED_COLS
)

# VectorAssembler: concatena todas as features em um único vetor
assembler = VectorAssembler(
    inputCols=ENCODED_COLS + NUMERIC_COLS,
    outputCol="features"
)

# Estágios base compartilhados por todos os modelos
BASE_STAGES = indexers + [encoder, assembler]

### 3.2 Divisão Treino / Teste e Cálculo do Tamanho do Vetor de Features

In [ ]:
train_data, test_data = df_balanced.randomSplit([0.8, 0.2], seed=42)
print(f"Treino: {train_data.count():,} | Teste: {test_data.count():,}")

# Descobre o tamanho do vetor de features (necessário para configurar o MLP)
pre_pipeline  = Pipeline(stages=BASE_STAGES)
pre_model     = pre_pipeline.fit(train_data)
sample_vector = pre_model.transform(train_data.limit(1))
INPUT_SIZE    = len(sample_vector.select("features").first()[0])

print(f"Tamanho do vetor de features: {INPUT_SIZE}")

### 3.3 Função Auxiliar de Avaliação

In [ ]:
def evaluate_model(predictions, label_col=TARGET_COL):
    """Extrai Accuracy, Precision, Recall e F1-Score via MulticlassClassificationEvaluator."""
    evaluator = MulticlassClassificationEvaluator(labelCol=label_col)
    metrics = {}
    for metric in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
        evaluator.setMetricName(metric)
        metrics[metric] = round(evaluator.evaluate(predictions), 4)
    return metrics

### 3.4 Modelo 1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(
    labelCol=TARGET_COL,
    featuresCol="features",
    maxIter=10
)

pipeline_lr = Pipeline(stages=BASE_STAGES + [lr])
model_lr    = pipeline_lr.fit(train_data)
pred_lr     = model_lr.transform(test_data)

metrics_lr = evaluate_model(pred_lr)
print("=== Logistic Regression ===")
for k, v in metrics_lr.items():
    print(f"  {k}: {v}")

### 3.5 Modelo 2 — Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(
    labelCol=TARGET_COL,
    featuresCol="features",
    maxDepth=5
)

pipeline_dt = Pipeline(stages=BASE_STAGES + [dt])
model_dt    = pipeline_dt.fit(train_data)
pred_dt     = model_dt.transform(test_data)

metrics_dt = evaluate_model(pred_dt)
print("=== Decision Tree Classifier ===")
for k, v in metrics_dt.items():
    print(f"  {k}: {v}")

### 3.6 Modelo 3 — Multilayer Perceptron (Rede Neural)

In [ ]:
# Arquitetura: [input, 12, 8, 2 (classes)]
# INPUT_SIZE calculado dinamicamente na célula 3.2
MLP_LAYERS = [INPUT_SIZE, 12, 8, 2]
print(f"Camadas MLP: {MLP_LAYERS}")

mlp = MultilayerPerceptronClassifier(
    labelCol=TARGET_COL,
    featuresCol="features",
    layers=MLP_LAYERS,
    maxIter=50,
    seed=42
)

pipeline_mlp = Pipeline(stages=BASE_STAGES + [mlp])
model_mlp    = pipeline_mlp.fit(train_data)
pred_mlp     = model_mlp.transform(test_data)

metrics_mlp = evaluate_model(pred_mlp)
print("=== MLP Neural Network ===")
for k, v in metrics_mlp.items():
    print(f"  {k}: {v}")

### 3.7 Comparação Final dos Modelos

In [ ]:
results = pd.DataFrame({
    "Model":     ["Logistic Regression", "Decision Tree", "MLP Neural Network"],
    "Accuracy":  [metrics_lr["accuracy"],          metrics_dt["accuracy"],          metrics_mlp["accuracy"]],
    "Precision": [metrics_lr["weightedPrecision"],  metrics_dt["weightedPrecision"],  metrics_mlp["weightedPrecision"]],
    "Recall":    [metrics_lr["weightedRecall"],      metrics_dt["weightedRecall"],      metrics_mlp["weightedRecall"]],
    "F1-Score":  [metrics_lr["f1"],                 metrics_dt["f1"],                 metrics_mlp["f1"]],
})

results.set_index("Model", inplace=True)
results

### 3.8 Persistência — Salvar Modelo e Dataset Processado

In [ ]:
# Salva o Decision Tree como modelo persistido
MODEL_PATH   = "/home/jovyan/work/data_parquet/model_decision_tree"
BALANCED_PATH = "/home/jovyan/work/data_parquet/crimes_balanced"

model_dt.write().overwrite().save(MODEL_PATH)
df_balanced.write.mode("overwrite").parquet(BALANCED_PATH)

print(f"Modelo salvo em:  {MODEL_PATH}")
print(f"Dataset salvo em: {BALANCED_PATH}")

In [ ]:
spark.stop()
print("SparkSession encerrada.")

---
## Relatório de Conclusões

| Aspecto | Observação |
|---|---|
| **Melhor Accuracy** | Decision Tree ou MLP (resultado a preencher após execução) |
| **Melhor F1-Score** | Indica o modelo mais equilibrado entre Precision e Recall |
| **Logistic Regression** | Baseline linear — tende a ter menor accuracy mas maior interpretabilidade |
| **Decision Tree** | Captura não-linearidades; risco de overfitting com `maxDepth` alto |
| **MLP** | Maior capacidade expressiva; requer mais dados e tempo de treino |
| **Balanceamento** | Undersampling reduziu o viés da classe majoritária (Arrest=0) |
| **Parquet vs CSV** | Parquet reduz I/O em ~70% para leituras colunar-seletivas no Spark |